# Week 4 Baseline First

This notebook compares customer intent classification baselines on the same dataset, split, metric and information boundary.

In [1]:
from pathlib import Path
import sys
import time

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))
from rule_baseline import predict_batch

DATA_PATH = ROOT / "data" / "customer_intent_demo.csv"
TARGET = "intent"
TEXT_FIELD = "message"
HIGH_RISK_LABEL = "refund_return"
RANDOM_STATE = 42

print("Repo root:", ROOT)
print("Dataset path:", DATA_PATH)

Repo root: C:\Users\yiting\Desktop\NCHU\1151\AI\ai-system-week1-starter
Dataset path: C:\Users\yiting\Desktop\NCHU\1151\AI\ai-system-week1-starter\data\customer_intent_demo.csv


In [2]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print("label distribution:")
print(df[TARGET].value_counts().sort_index().to_string())

shape: (100, 2)
label distribution:
intent
account_login     25
order_delivery    25
product_info      25
refund_return     25


In [3]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df[TARGET],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df[TARGET],
)

print("train rows:", len(train_df))
print("validation rows:", len(val_df))
print("test rows:", len(test_df))
print("test label distribution:")
print(test_df[TARGET].value_counts().sort_index().to_string())

train rows: 70
validation rows: 15
test rows: 15
test label distribution:
intent
account_login     4
order_delivery    4
product_info      4
refund_return     3


In [4]:
def evaluate_predictions(method, pred, elapsed, complexity, note):
    y_true = test_df[TARGET]
    return {
        "method": method,
        "dataset_information": "message only; stratified random split seed=42",
        "accuracy": accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "refund_return_recall": recall_score(y_true, pred, labels=[HIGH_RISK_LABEL], average="macro", zero_division=0),
        "runtime_seconds": elapsed,
        "complexity": complexity,
        "note": note,
    }

predictions = {}
results = []

start = time.perf_counter()
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(train_df[[TEXT_FIELD]], train_df[TARGET])
pred = dummy.predict(test_df[[TEXT_FIELD]])
results.append(evaluate_predictions("Majority Class", pred, time.perf_counter() - start, "Low", "Naive baseline"))
predictions["Majority Class"] = pred

start = time.perf_counter()
pred = predict_batch(test_df[TEXT_FIELD].tolist())
results.append(evaluate_predictions("Rule-based Keyword", pred, time.perf_counter() - start, "Low", "Current credible baseline"))
predictions["Rule-based Keyword"] = pred

start = time.perf_counter()
logistic_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])
logistic_pipeline.fit(train_df[TEXT_FIELD], train_df[TARGET])
pred = logistic_pipeline.predict(test_df[TEXT_FIELD])
results.append(evaluate_predictions("TF-IDF Logistic", pred, time.perf_counter() - start, "Low-Mid", "Simple ML candidate"))
predictions["TF-IDF Logistic"] = pred

results_df = pd.DataFrame(results)
for col in ["accuracy", "macro_f1", "refund_return_recall", "runtime_seconds"]:
    results_df[col] = results_df[col].round(4)
print(results_df.to_string(index=False))

            method                           dataset_information  accuracy  macro_f1  refund_return_recall  runtime_seconds complexity                                                       note
    Majority Class message only; stratified random split seed=42    0.2667    0.1053                0.0000           0.0029        Low Naive baseline; predicts the most frequent training label.
Rule-based Keyword message only; stratified random split seed=42    1.0000    1.0000                1.0000           0.0005        Low Current credible baseline from Week 1; uses keyword rules.
   TF-IDF Logistic message only; stratified random split seed=42    0.7333    0.6984                0.3333           0.0340    Low-Mid   Simple ML candidate; preprocessing is fit on train only.


In [5]:
failure_rows = []
for method, pred in predictions.items():
    temp = test_df.copy().reset_index().rename(columns={"index": "case_id"})
    temp["prediction"] = pred
    wrong = temp[temp[TARGET] != temp["prediction"]].copy()
    for _, row in wrong.iterrows():
        failure_rows.append(
            {
                "method": method,
                "case_id": int(row["case_id"]),
                "message": row[TEXT_FIELD],
                "ground_truth": row[TARGET],
                "prediction": row["prediction"],
            }
        )

failure_df = pd.DataFrame(failure_rows)
print("failure rows:", len(failure_df))
print(failure_df.head(10).to_string(index=False))

failure rows: 15
        method  case_id     message  ground_truth     prediction          failure_type                                                     possible_cause                                                                decision_impact
Majority Class       74 會員生日資料可以修改嗎 account_login order_delivery single-class collapse The method ignores text and sends every message to the same label.                              May route the customer to the wrong support flow.
Majority Class       36 發票開錯想退費可以改嗎 refund_return order_delivery single-class collapse The method ignores text and sends every message to the same label. High risk because refund or return messages may miss the refund handling flow.
Majority Class       57    我想刪除會員帳號 account_login order_delivery single-class collapse The method ignores text and sends every message to the same label.                              May route the customer to the wrong support flow.
Majority Class       89     保固需要登錄嗎  product_info o

In [6]:
representative_failures = pd.concat(
    [
        failure_df[failure_df["method"].eq("TF-IDF Logistic")].head(3),
        failure_df[failure_df["method"].eq("Majority Class")].head(1),
    ],
    ignore_index=True,
).head(4)
print(representative_failures.to_string(index=False))

         method  case_id         message   ground_truth     prediction          failure_type                                                                         possible_cause                                                                decision_impact
TF-IDF Logistic       36     發票開錯想退費可以改嗎  refund_return order_delivery  short-text ambiguity The small synthetic training set does not provide enough evidence for similar wording. High risk because refund or return messages may miss the refund handling flow.
TF-IDF Logistic       14 我想查訂單編號A1023的進度 order_delivery  refund_return  short-text ambiguity The small synthetic training set does not provide enough evidence for similar wording.                              May route the customer to the wrong support flow.
TF-IDF Logistic       28      請問七天鑑賞期怎麼算  refund_return   product_info  short-text ambiguity The small synthetic training set does not provide enough evidence for similar wording. High risk because refund or return messages

In [7]:
outputs = ROOT / "outputs"
outputs.mkdir(exist_ok=True)
results_df.to_csv(outputs / "week04_results.csv", index=False, encoding="utf-8-sig")
failure_df.to_csv(outputs / "week04_failure_cases.csv", index=False, encoding="utf-8-sig")
print("wrote:", outputs / "week04_results.csv")
print("wrote:", outputs / "week04_failure_cases.csv")

wrote: C:\Users\yiting\Desktop\NCHU\1151\AI\ai-system-week1-starter\outputs\week04_results.csv
wrote: C:\Users\yiting\Desktop\NCHU\1151\AI\ai-system-week1-starter\outputs\week04_failure_cases.csv
